# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder: OPTIONAL to execute C++ code or Rust code</h2>
            <span style="color:#f71;">As an alternative, you can run it on the website given yesterday</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            In this lab, I use high end models GPT 5, Claude 4.5 Sonnet, Gemini 2.5 Pro, Grok 4, which are the slightly higher priced models. The costs are still low, but if you'd prefer to keep costs ultra low, please pick lower cost models like gpt-5-nano.
            </span>
        </td>
    </tr>
</table>

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
import gradio as gr
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)

google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

print("Checking API Keys...")
print(f"Gemini: {'✅ Loaded' if google_api_key else '❌ Missing'}")
print(f"Groq: {'✅ Loaded' if groq_api_key else '❌ Missing'}")
print(f"OpenRouter: {'✅ Loaded' if openrouter_api_key else '❌ Missing'}")

Checking API Keys...
Gemini: ✅ Loaded
Groq: ✅ Loaded
OpenRouter: ✅ Loaded


In [3]:
# 1. Setup Gemini Client (Google)
gemini = OpenAI(
    api_key=google_api_key, 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

# 2. Setup Groq Client
groq = OpenAI(
    api_key=groq_api_key, 
    base_url="https://api.groq.com/openai/v1"
)

# 3. Setup OpenRouter Client
openrouter = OpenAI(
    api_key=openrouter_api_key, 
    base_url="https://openrouter.ai/api/v1"
)

In [4]:
# The absolute best free models on each platform right now:
GEMINI_MODEL = "gemini-2.5-flash"
GROQ_MODEL = "llama-3.3-70b-versatile"
OPENROUTER_MODEL = "mistralai/mistral-7b-instruct:free" # Swapped to Mistral for stability!

# Create a list for our Gradio dropdown
models = [GEMINI_MODEL, GROQ_MODEL, OPENROUTER_MODEL]

# Map each model name to the correct client
clients = {
    GEMINI_MODEL: gemini,
    GROQ_MODEL: groq,
    OPENROUTER_MODEL: openrouter
}

print(f"Loaded {len(models)} free models ready for generation!")

Loaded 3 free models ready for generation!


In [5]:
from system_info import retrieve_system_info, rust_toolchain_info

system_info = retrieve_system_info()
rust_info = rust_toolchain_info()
rust_info

{'installed': True,
 'rustc': {'path': 'C:\\Users\\HP\\.cargo\\bin\\rustc.EXE',
  'version': '',
  'host_triple': '',
  'release': '',
  'commit_hash': ''},
 'cargo': {'path': 'C:\\Users\\HP\\.cargo\\bin\\cargo.EXE', 'version': ''},
 'rustup': {'path': 'C:\\Users\\HP\\.cargo\\bin\\rustup.EXE',
  'version': 'rustup 1.29.0 (28d1352db 2026-03-05)',
  'active_toolchain': '',
  'default_toolchain': '',
  'toolchains': ['no installed toolchains'],
  'targets_installed': []},
 'rust_analyzer': {'path': 'C:\\Users\\HP\\.cargo\\bin\\rust-analyzer.EXE'},
 'env': {'CARGO_HOME': 'C:\\Users\\HP\\.cargo',
  'RUSTUP_HOME': 'C:\\Users\\HP\\.rustup',
  'RUSTFLAGS': '',
  'CARGO_BUILD_TARGET': ''},
 'execution_examples': ['"C:\\Users\\HP\\.cargo\\bin\\cargo.EXE" build',
  '"C:\\Users\\HP\\.cargo\\bin\\cargo.EXE" run',
  '"C:\\Users\\HP\\.cargo\\bin\\cargo.EXE" test',
  '"C:\\Users\\HP\\.cargo\\bin\\rustc.EXE" hello.rs -o hello']}

In [6]:
message = f"""
Here is a report of the system information for my computer.
I want to run a Rust compiler to compile a single rust file called main.rs and then execute it in the simplest way possible.
Please reply with whether I need to install a Rust toolchain to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile Rust code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.
Have the maximum possible runtime performance in mind; compile time can be slow. Fastest possible runtime performance for this platform is key.
Reply with the commands in markdown.

System information:
{system_info}

Rust toolchain information:
{rust_info}
"""

response = gemini.chat.completions.create(model=GEMINI_MODEL, messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))

Based on the `Rust toolchain information` provided, although `rustup` is installed, it explicitly states `'toolchains': ['no installed toolchains']`. This means that while the Rust toolchain *manager* (`rustup`) is present, no actual Rust compiler (`rustc`) or package manager (`cargo`) has been installed for use.

Therefore, **yes, you need to install a Rust toolchain** to compile and execute Rust code.

Here are the simplest step-by-step instructions to do so:

### Step-by-Step Instructions to Install Rust Toolchain

1.  **Open a Command Prompt or PowerShell:**
    Search for "cmd" or "PowerShell" in your Windows search bar and open it.

2.  **Install the stable Rust toolchain:**
    Run the following command. `rustup` will download and install the latest stable version of the Rust compiler and its associated tools.
    ```cmd
    rustup install stable
    ```
    This command might take a few minutes as it downloads the toolchain components.

3.  **Set the stable toolchain as default:**
    After installation, ensure the `stable` toolchain is set as the default one that `rustup` will use.
    ```cmd
    rustup default stable
    ```

4.  **Verify the installation:**
    To confirm that `rustc` and `cargo` are now correctly installed and accessible, run these commands:
    ```cmd
    rustc --version
    cargo --version
    ```
    You should see version information displayed for both, indicating a successful installation. The `C:\Users\HP\.cargo\bin` directory, where these executables are located, should have been added to your system's `PATH` environment variable during the initial `rustup` installation.

---

Once your Rust toolchain is set up, you can use the following Python `subprocess` commands to compile and execute your `main.rs` file for the fastest possible runtime performance:

For the fastest possible runtime performance, we will use `rustc` directly with maximum optimization flags:
*   `-C opt-level=3`: This is the highest optimization level, instructing the compiler to perform aggressive optimizations.
*   `-C lto=true`: Enables Link Time Optimization, which allows the compiler to optimize across different compilation units, potentially leading to better overall performance.
*   `-C codegen-units=1`: Forces the compiler to generate a single code generation unit. While this increases compile time (as it prevents parallel compilation of parts of your code), it allows for more extensive inter-procedural optimizations, which can yield faster execution.

```python
import subprocess
import os

# Assuming main.rs is in the current working directory
# For demonstration, let's create a dummy main.rs if it doesn't exist
# You should replace this with your actual main.rs file
if not os.path.exists("main.rs"):
    with open("main.rs", "w") as f:
        f.write("""
fn main() {
    println!("Hello from Rust!");
}
""")

# Compile command:
# Compiles main.rs to main.exe with maximum runtime optimizations.
# -o main.exe specifies the output executable name.
compile_command = [
    "rustc",
    "main.rs",
    "-o", "main.exe",
    "-C", "opt-level=3",
    "-C", "lto=true",
    "-C", "codegen-units=1"
]

# Run command:
# Executes the compiled Rust program.
# On Windows, specifying "./main.exe" ensures it runs from the current directory.
run_command = ["./main.exe"]

# Example of how you would integrate this into your Python script:
try:
    print(f"Compiling with: {' '.join(compile_command)}")
    compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
    print("Compilation successful.")
    if compile_result.stdout:
        print("Compile stdout:", compile_result.stdout.strip())
    if compile_result.stderr:
        print("Compile stderr:", compile_result.stderr.strip())

    print(f"\nRunning with: {' '.join(run_command)}")
    run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
    print("Execution successful.")
    # The return value for your function as requested
    print("Run stdout:", run_result.stdout.strip())
    if run_result.stderr:
        print("Run stderr:", run_result.stderr.strip())

except subprocess.CalledProcessError as e:
    print(f"Error during command execution: {e}")
    print(f"Stdout: {e.stdout}")
    print(f"Stderr: {e.stderr}")
except FileNotFoundError:
    print("Error: rustc or the compiled executable not found.")
    print("Please ensure Rust is installed and added to your system PATH, or that the executable exists.")

# Clean up the dummy main.rs and main.exe (optional)
# if os.path.exists("main.rs") and not os.path.exists("main.rs.original"): # prevent deleting user file
#     os.remove("main.rs")
# if os.path.exists("main.exe"):
#     os.remove("main.exe")
```

## For C++, overwrite this with the commands from yesterday, or for Rust, use the new commands

Or just use the website like yesterday:

 https://www.programiz.com/cpp-programming/online-compiler/

In [17]:
# Check which language we are targeting
if language == "C++":
    compile_command = ["g++", "main.cpp", "-o", "main.exe", "-O3"]
    run_command = ["./main.exe"]
elif language == "Rust":
    compile_command = [
        "rustc", 
        "main.rs", 
        "-o", "main.exe",
        "-C", "opt-level=3", 
        "-C", "lto=true", 
        "-C", "codegen-units=1"
    ]
    run_command = ["./main.exe"]

## And now, on with the main task

In [8]:
language = "Rust" # or "C++"
extension = "rs" if language == "Rust" else "cpp"

system_prompt = f"""
Your task is to convert Python code into high performance {language} code.
Respond only with {language} code. Do not provide any explanation other than occasional comments.
The {language} response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to {language} with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.{language} and then compiled and executed; the compilation command is:
{compile_command}
Respond only with {language} code.
Python code to port:

```python
{python}
```
"""

In [9]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [10]:
def write_output(code):
    with open(f"main.{extension}", "w") as f:
        f.write(code)

In [11]:
def port(model, python):
    # Grab the correct client (Gemini, Groq, or OpenRouter)
    client = clients[model]
    
    # Use messages_for() so it correctly asks for Rust OR C++ based on your setting
    response = client.chat.completions.create(
        model=model, 
        messages=messages_for(python)
    )
    
    reply = response.choices[0].message.content
    
    # Clean up markdown tags whether it outputs C++ or Rust
    reply = reply.replace('```cpp\n', '').replace('```cpp', '')
    reply = reply.replace('```rust\n', '').replace('```rust', '')
    reply = reply.replace('```', '')
    
    return reply

In [12]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [13]:
# Check which language we are targeting
if language == "C++":
    compile_command = ["g++", "main.cpp", "-o", "main.exe", "-O3"]
    run_command = ["./main.exe"]
elif language == "Rust":
    # Make sure you installed Rust on Windows for this to work!
    compile_command = [
        "rustc", 
        "main.rs", 
        "-C", "opt-level=3", 
        "-C", "target-cpu=native", 
        "-o", "main.exe"
    ]
    run_command = ["./main.exe"]

def compile_and_run(code):
    write_output(code)
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
        return run_result.stdout
    except subprocess.CalledProcessError as e:
        return f"🚨 Compilation Error:\n{e.stderr}"
    except FileNotFoundError as e:
        return f"🚨 Compiler Not Found! Make sure {compile_command[0]} is installed and on your Windows PATH.\nError: {e}"

In [18]:
python_hard = """# Be careful to support large numbers

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 1000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [19]:
from styles import CSS

with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Port from Python to {language}") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label="Python (original)",
                value=python_hard,
                language="python",
                lines=26
            )
        with gr.Column(scale=6):
            cpp = gr.Code(
                label=f"{language} (generated)",
                value="",
                language="cpp",
                lines=26
            )

    with gr.Row(elem_classes=["controls"]):
        python_run = gr.Button("Run Python", elem_classes=["run-btn", "py"])
        model = gr.Dropdown(models, value=models[0], show_label=False)
        convert = gr.Button(f"Port to {language}", elem_classes=["convert-btn"])
        cpp_run = gr.Button(f"Run {language}", elem_classes=["run-btn", "cpp"])

    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_out = gr.TextArea(label="Python result", lines=8, elem_classes=["py-out"])
        with gr.Column(scale=6):
            cpp_out = gr.TextArea(label=f"{language} result", lines=8, elem_classes=["cpp-out"])

    convert.click(fn=port, inputs=[model, python], outputs=[cpp])
    python_run.click(fn=run_python, inputs=[python], outputs=[python_out])
    cpp_run.click(fn=compile_and_run, inputs=[cpp], outputs=[cpp_out])

ui.launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\Users\HP\projects\llm_engineering\.venv\Lib\site-packages\gradio\queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\HP\projects\llm_engineering\.venv\Lib\site-packages\gradio\route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\HP\projects\llm_engineering\.venv\Lib\site-packages\gradio\blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\HP\projects\llm_engineering\.venv\Lib\site-packages\gradio\blocks.py", line 1623, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\HP\projects\llm_engineering\.venv\Lib\site-packages\anyio\to_thread.p

## RESULTS!

mistralai/mistral-7b-instruct:free: FAIL   

**2nd place: llama-3.3-70b-versatile: 0.023621700**

**1st place: Gemini 2.5 Pro: 0.013784**  